In [40]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from google.colab import files

# Upload your original CSV
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Saving Hospital_Dataset.csv to Hospital_Dataset (2).csv
Dataset loaded successfully!
Rows: 5012
Columns: 42


In [41]:
print("All columns in the dataset:\n")

for i, col in enumerate(df.columns):
    print(i + 1, ":", repr(col))

All columns in the dataset:

1 : 'Case_No'
2 : 'Hospital_ID'
3 : 'Specialty_Admission'
4 : 'Admission_Date'
5 : 'Admission_Time'
6 : 'Discharge_Date'
7 : 'Discharge_Time'
8 : 'LOS_Admission'
9 : 'Readmitted'
10 : 'Transfer_From'
11 : 'Transfer_To'
12 : 'Month'
13 : 'DOB'
14 : 'Nationality'
15 : 'Gender'
16 : 'DoctorLicense'
17 : 'DoctorName'
18 : 'Doctor Type'
19 : 'Doctor Status'
20 : 'CMI Value'
21 : 'Specialty_Generated'
22 : 'Insurance/Payer'
23 : 'InsurancePlanName'
24 : 'Payer Mix'
25 : 'Case type'
26 : 'LOS_Generated'
27 : 'Severity'
28 : 'Surgical Mix'
29 : 'Discharge Time'
30 : 'Discharge Before 12PM'
31 : 'Revenue'
32 : 'Specialty'
33 : 'Case_Count'
34 : 'Average_LOS'
35 : 'Total_LOS_Days'
36 : 'Total_Beds'
37 : 'Occupied_Beds'
38 : 'Available_Beds'
39 : 'Occupancy_Rate'
40 : 'Doctors_Count'
41 : 'Nurses_Count'
42 : 'Other_Staff_Count'


In [42]:
print("========== DATASET PROFILE ==========")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== MISSING VALUES ==========")

missing = df.isnull().sum()

missing_report = pd.DataFrame({
    "Column": missing.index,
    "Missing_Count": missing.values,
    "Missing_Percentage": (
        missing.values / len(df) * 100
    ).round(2)
})

print(
    missing_report[
        missing_report["Missing_Count"] > 0
    ].to_string(index=False)
)

========== DATASET PROFILE ==========
Rows: 5012
Columns: 42

========== DATA TYPES ==========
Case_No                    int64
Hospital_ID               object
Specialty_Admission       object
Admission_Date            object
Admission_Time            object
Discharge_Date            object
Discharge_Time            object
LOS_Admission            float64
Readmitted                object
Transfer_From             object
Transfer_To               object
Month                     object
DOB                      float64
Nationality               object
Gender                    object
DoctorLicense             object
DoctorName                object
Doctor Type               object
Doctor Status             object
CMI Value                float64
Specialty_Generated       object
Insurance/Payer           object
InsurancePlanName         object
Payer Mix                 object
Case type                 object
LOS_Generated            float64
Severity                 float64
Surgical Mix  

In [43]:
print("Rows before duplicate removal:", len(df))

duplicate_count = df["Case_No"].duplicated().sum()

print("Duplicate Case_No records:", duplicate_count)

df = df.drop_duplicates(
    subset=["Case_No"],
    keep="first"
).copy()

print("Rows after duplicate removal:", len(df))
print("Unique Case_No:", df["Case_No"].nunique())

Rows before duplicate removal: 5012
Duplicate Case_No records: 12
Rows after duplicate removal: 5000
Unique Case_No: 5000


In [44]:
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string").str.strip()

print("Text columns cleaned.")

Text columns cleaned.


In [45]:
categorical_columns = [
    "Gender",
    "Nationality",
    "Transfer_To",
    "Readmitted",
    "Insurance/Payer",
    "Case type",
    "DoctorName",
    "Doctor Status",
    "InsurancePlanName",
    "Payer Mix",
    "Surgical Mix"
]

for col in categorical_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

        if col == "DoctorName":
            df[col] = df[col].fillna("Unassigned")
        else:
            df[col] = df[col].fillna("Unknown")

print("Categorical missing values handled.")

Categorical missing values handled.


In [46]:
if "Gender" in df.columns:

    df["Gender"] = (
        df["Gender"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    gender_mapping = {
        "F": "Female",
        "FEMALE": "Female",
        "WOMAN": "Female",

        "M": "Male",
        "MALE": "Male",
        "MAN": "Male",

        "UNKNOWN": "Unknown"
    }

    df["Gender"] = df["Gender"].replace(gender_mapping)

    print(df["Gender"].value_counts(dropna=False))

Gender
Male       2525
Female     2400
Unknown      75
Name: count, dtype: Int64


In [47]:
specialty_columns = [
    "Specialty_Admission",
    "Specialty_Generated",
    "Specialty"
]

for col in specialty_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(
                r"\s+",
                " ",
                regex=True
            )
            .str.title()
        )

print("Department names standardized.")

Department names standardized.


In [48]:
specialty_mapping = {

    "Orthopedic": "Orthopaedics",
    "Orthopedics": "Orthopaedics",
    "Orthopaedic": "Orthopaedics",

    "General Surgery": "General Surgery",

    "Internal Medicine": "Internal Medicine",

    "Emergency Medicine": "Emergency",

    "Obstetrics And Gynecology":
        "Obstetrics & Gynecology"
}

for col in specialty_columns:

    if col in df.columns:
        df[col] = df[col].replace(
            specialty_mapping
        )

print("Department variations standardized.")

Department variations standardized.


In [49]:
date_columns = [
    "Admission_Date",
    "Discharge_Date",
    "Month"
]

for col in date_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            format="mixed",
            errors="coerce"
        )

print("Date conversion completed.")

for col in date_columns:
    if col in df.columns:
        print(col, ":", df[col].dtype)

Date conversion completed.
Admission_Date : datetime64[ns]
Discharge_Date : datetime64[ns]
Month : datetime64[ns]


In [50]:
time_columns = [
    "Admission_Time",
    "Discharge_Time",
    "Discharge Time"
]

for col in time_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col].astype("string"),
            format="mixed",
            errors="coerce"
        ).dt.time

print("Time conversion completed.")

Time conversion completed.


In [51]:
df["Discharge_Time_Final"] = (
    df["Discharge_Time"]
    .fillna(df["Discharge Time"])
)

print(
    "Missing discharge times:",
    df["Discharge_Time_Final"].isna().sum()
)

print(
    "Valid discharge times:",
    df["Discharge_Time_Final"].notna().sum()
)

Missing discharge times: 3
Valid discharge times: 4997


In [52]:
if "Severity" in df.columns:

    df["Severity"] = pd.to_numeric(
        df["Severity"],
        errors="coerce"
    )

    severity_median = df["Severity"].median()

    df["Severity"] = df["Severity"].fillna(
        severity_median
    )

    print("Severity median:", severity_median)
    print(
        "Missing Severity:",
        df["Severity"].isna().sum()
    )

Severity median: 2.0
Missing Severity: 0


In [53]:
numeric_columns = [
    "CMI Value",
    "LOS_Admission",
    "LOS_Generated",
    "Revenue",
    "Case_Count",
    "Average_LOS",
    "Total_LOS_Days",
    "Total_Beds",
    "Occupied_Beds",
    "Available_Beds",
    "Occupancy_Rate",
    "Doctors_Count",
    "Nurses_Count",
    "Other_Staff_Count"
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

print("Numerical columns converted.")

Numerical columns converted.


In [54]:
if all(
    col in df.columns
    for col in [
        "Admission_Date",
        "Discharge_Date",
        "LOS_Admission"
    ]
):

    missing_discharge_date = (
        df["Discharge_Date"].isna()
        &
        df["Admission_Date"].notna()
        &
        df["LOS_Admission"].notna()
    )

    df.loc[
        missing_discharge_date,
        "Discharge_Date"
    ] = (
        df.loc[
            missing_discharge_date,
            "Admission_Date"
        ]
        +
        pd.to_timedelta(
            df.loc[
                missing_discharge_date,
                "LOS_Admission"
            ],
            unit="D"
        )
    )

print(
    "Missing Discharge_Date:",
    df["Discharge_Date"].isna().sum()
)

Missing Discharge_Date: 2


In [55]:
valid_dates = (
    df["Admission_Date"].notna()
    &
    df["Discharge_Date"].notna()
)

calculated_los = (
    df["Discharge_Date"]
    -
    df["Admission_Date"]
).dt.days

df.loc[
    valid_dates,
    "LOS_Admission"
] = calculated_los[valid_dates]

# Remove impossible negative LOS
df.loc[
    df["LOS_Admission"] < 0,
    "LOS_Admission"
] = np.nan

print(df["LOS_Admission"].describe())

count    4927.000000
mean        3.479399
std        20.528965
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max       326.000000
Name: LOS_Admission, dtype: float64


In [56]:
if "Revenue" in df.columns:

    # Convert invalid negative revenue to missing
    df.loc[
        df["Revenue"] < 0,
        "Revenue"
    ] = np.nan

    revenue_median = df["Revenue"].median()

    df["Revenue"] = df["Revenue"].fillna(
        revenue_median
    )

    print("Revenue median:", revenue_median)

    print(
        "Missing Revenue:",
        df["Revenue"].isna().sum()
    )

Revenue median: 2107.49
Missing Revenue: 0


In [57]:
required_bed_columns = [
    "Total_Beds",
    "Occupied_Beds",
    "Available_Beds"
]

if all(
    col in df.columns
    for col in required_bed_columns
):

    missing_occupied = (
        df["Occupied_Beds"].isna()
        &
        df["Total_Beds"].notna()
        &
        df["Available_Beds"].notna()
    )

    df.loc[
        missing_occupied,
        "Occupied_Beds"
    ] = (
        df.loc[
            missing_occupied,
            "Total_Beds"
        ]
        -
        df.loc[
            missing_occupied,
            "Available_Beds"
        ]
    )

    df["Occupied_Beds"] = (
        df["Occupied_Beds"].clip(lower=0)
    )

print(
    "Missing Occupied_Beds:",
    df["Occupied_Beds"].isna().sum()
)

Missing Occupied_Beds: 0


In [58]:
valid_bed_records = (
    df["Total_Beds"].notna()
    &
    (df["Total_Beds"] > 0)
    &
    df["Occupied_Beds"].notna()
)

df.loc[
    valid_bed_records,
    "Occupancy_Rate"
] = (
    df.loc[
        valid_bed_records,
        "Occupied_Beds"
    ]
    /
    df.loc[
        valid_bed_records,
        "Total_Beds"
    ]
) * 100

df["Occupancy_Rate"] = (
    df["Occupancy_Rate"]
    .clip(lower=0, upper=100)
)

print(
    df["Occupancy_Rate"].describe()
)

count    5000.000000
mean       64.890730
std        21.215482
min         0.000000
25%        60.000000
50%        70.000000
75%        80.000000
max       100.000000
Name: Occupancy_Rate, dtype: float64


In [59]:
for col in [
    "Doctors_Count",
    "Nurses_Count"
]:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if "Hospital_ID" in df.columns:

            df[col] = (
                df.groupby("Hospital_ID")[col]
                .transform(
                    lambda x: x.fillna(
                        x.median()
                    )
                )
            )

        # Overall median fallback
        df[col] = df[col].fillna(
            df[col].median()
        )

print(
    "Missing Doctors_Count:",
    df["Doctors_Count"].isna().sum()
)

print(
    "Missing Nurses_Count:",
    df["Nurses_Count"].isna().sum()
)

Missing Doctors_Count: 0
Missing Nurses_Count: 0


In [60]:
discharge_datetime = pd.to_datetime(
    df["Discharge_Time_Final"].astype("string"),
    format="mixed",
    errors="coerce"
)

df["Discharge Before 12PM"] = np.where(
    discharge_datetime.isna(),
    "Unknown",
    np.where(
        discharge_datetime.dt.hour < 12,
        "Yes",
        "No"
    )
)

print(
    df["Discharge Before 12PM"]
    .value_counts(dropna=False)
)

Discharge Before 12PM
No         3095
Yes        1902
Unknown       3
Name: count, dtype: int64


In [61]:
healthcare_indicators = [
    "LOS_Admission",
    "Revenue",
    "Occupancy_Rate",
    "CMI Value",
    "Occupied_Beds",
    "Doctors_Count",
    "Nurses_Count",
    "Other_Staff_Count"
]

available_indicators = [
    col
    for col in healthcare_indicators
    if col in df.columns
]

scaler = MinMaxScaler()

df_normalized = df[
    available_indicators
].copy()

# Fill temporary missing values only for scaling
df_normalized = df_normalized.fillna(
    df_normalized.median()
)

normalized_values = scaler.fit_transform(
    df_normalized
)

for i, col in enumerate(
    available_indicators
):

    df[col + "_Normalized"] = (
        normalized_values[:, i]
    )

print("Healthcare indicators normalized.")

Healthcare indicators normalized.


In [62]:
missing = df.isnull().sum()

final_missing_report = pd.DataFrame({
    "Column": missing.index,
    "Missing_Count": missing.values,
    "Missing_Percentage": (
        missing.values / len(df) * 100
    ).round(2)
})

print(
    final_missing_report[
        final_missing_report["Missing_Count"] > 0
    ].to_string(index=False)
)

              Column  Missing_Count  Missing_Percentage
      Admission_Time             75                1.50
      Discharge_Date              2                0.04
      Discharge_Time             90                1.80
       LOS_Admission             73                1.46
      Discharge Time            237                4.74
Discharge_Time_Final              3                0.06


In [63]:
print("========== FINAL DUPLICATE CHECK ==========")

print(
    "Total rows:",
    len(df)
)

print(
    "Unique Case_No:",
    df["Case_No"].nunique()
)

print(
    "Duplicate Case_No:",
    df["Case_No"].duplicated().sum()
)

========== FINAL DUPLICATE CHECK ==========
Total rows: 5000
Unique Case_No: 5000
Duplicate Case_No: 0


In [64]:
print("========== FINAL DATASET ==========")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")

for i, col in enumerate(df.columns):
    print(i + 1, col)


========== FINAL DATASET ==========
Rows: 5000
Columns: 51

Column names:
1 Case_No
2 Hospital_ID
3 Specialty_Admission
4 Admission_Date
5 Admission_Time
6 Discharge_Date
7 Discharge_Time
8 LOS_Admission
9 Readmitted
10 Transfer_From
11 Transfer_To
12 Month
13 DOB
14 Nationality
15 Gender
16 DoctorLicense
17 DoctorName
18 Doctor Type
19 Doctor Status
20 CMI Value
21 Specialty_Generated
22 Insurance/Payer
23 InsurancePlanName
24 Payer Mix
25 Case type
26 LOS_Generated
27 Severity
28 Surgical Mix
29 Discharge Time
30 Discharge Before 12PM
31 Revenue
32 Specialty
33 Case_Count
34 Average_LOS
35 Total_LOS_Days
36 Total_Beds
37 Occupied_Beds
38 Available_Beds
39 Occupancy_Rate
40 Doctors_Count
41 Nurses_Count
42 Other_Staff_Count
43 Discharge_Time_Final
44 LOS_Admission_Normalized
45 Revenue_Normalized
46 Occupancy_Rate_Normalized
47 CMI Value_Normalized
48 Occupied_Beds_Normalized
49 Doctors_Count_Normalized
50 Nurses_Count_Normalized
51 Other_Staff_Count_Normalized


In [65]:
output_file = "Hospital_Dataset_Preprocessed.csv"

df.to_csv(
    output_file,
    index=False
)

print("Dataset successfully saved!")
print("File:", output_file)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset successfully saved!
File: Hospital_Dataset_Preprocessed.csv
Rows: 5000
Columns: 51


In [66]:
from google.colab import files

files.download(
    "Hospital_Dataset_Preprocessed.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>